# Technical Analysis and Narrative: Germany Energy Forecasting Prototype

This documentation provides the technical rationale behind the observed correlations within the German power market (2020–2026). By integrating high-resolution Quarter-hourly (QH) load data with multi-variable weather stream

## 1. Data Ingestion and Granularity: The QH Advantage

The transition from standard UK Non-half-hourly (NHH) forecasting to the German Quarter-hourly (QH - Every 15 mins) model allows for the capture of steep "ramp-up" and "ramp-down" periods.

**Technical Context:** High-resolution granularity is essential for managing the imbalance settlement process. Deviations from contracted positions must be settled at the System Buy/Sell Price. By analyzing QH data, we identify intra-hour volatility that hourly models smooth over, directly reducing the risk of imbalance charges.

**Commercial Logic:** This precision supports the "Smartest Short" strategy by allowing traders to optimize positions in 15-minute Intraday blocks rather than broader hourly aggregates.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import openmeteo_requests
from datetime import datetime
import requests_cache
from retry_requests import retry



CSV_FILE = "Actual_consumption_202001010000_202601010000_Quarterhour.csv"

def smard_csv(file_path):
    # Semicolon separator, comma = thousands, dot = decimal
    df = pd.read_csv(file_path, sep=';', thousands=',', decimal='.', index_col=False)

    df['datetime'] = pd.to_datetime(df['Start date'])
    df = df.set_index('datetime') # to change 0, 1, 2, index in to date itself
    
    load_col = 'grid load [MWh] Original resolutions'
    df = df[[load_col]].rename(columns={load_col: 'load_mwh'}) 
    
    return df

df_load = smard_csv(CSV_FILE)

df = df_load

df = df.dropna()
df['year'] = df.index.year
df['month'] = df.index.month
df['hour'] = df.index.hour
df['is_weekend'] = df.index.weekday >= 5
df['day_of_year'] = df.index.dayofyear

print(f"Data processed: {df.index.min()} to {df.index.max()}")
df.tail()


In [ ]:
import pandas as pd

CSV_FILE = "Actual_consumption_202001010000_202601010000_Quarterhour.csv"

def smard_csv(file_path):
    # Semicolon separator, comma = thousands, dot = decimal
    df = pd.read_csv(file_path, sep=';', thousands=',', decimal='.', index_col=False)
    return df
df_load = smard_csv(CSV_FILE)

df = df_load
df.head(100)


## 2. Average load profile

The daily load profile analysis highlights a dual-peak structure during the work week (Standard Load Profile — SLP).

**Technical Analysis:** The 8:00 AM peak represents the simultaneous activation of industrial processes and commercial lighting/HVAC. The gap between the Weekday and Weekend curves quantifies the Industrial Baseload Component.  

**The Findings:** During weekends, the load curve is not only lower but exhibits a different shape, characterized by a slower morning ramp. This suggests that residential demand (synthetic H0 profiles) is less sensitive to fixed start-times than the industrial RLM (Registering Load Measurement) demand.  

### Load Categorization
**Baseload**: The constant, non-varying minimum demand observed between 02:00 and 04:00 AM. This is the "floor" of the grid, driven by 24/7 industrial processes and essential infrastructure. 

**Peak**: The high-demand window (typically 08:00 to 20:00) where industrial RLM demand and commercial SLP demand coincide.  

**Off-Peak**: Periods of lower demand, primarily nights and weekends, where the "Industrial Baseload Component" is absent.  

### Anatomy of the Average Load Profile
**The Industrial Pulse (RLM)**: The sharp gradient at 08:00 AM is the technical signature of RLM (Registering Load Measurement) customers. These are large-scale consumers whose demand is metered in real-time. The steepness of this ramp requires high load-following capacity from conventional and flexible assets.  

**The Consumer Pulse (SLP)**: The secondary evening peak represents the SLP (Standard Load Profile), specifically the H0 (Household) and G0 (Small Business) profiles.  


### Grid Stability and Frequency Reserves (aFRR/aFFR)
Because electricity must be generated and consumed instantaneously, the 4 German TSOs (50Hertz, Amprion, TenneT, and TransnetBW) must manage any deviation between our forecasted load and actual QH consumption.  

**aFRR (Automatic Frequency Restoration Reserve)**: When the actual load deviates from our 15-minute schedule, the TSO automatically activates aFRR to restore the frequency to 50Hz.  

**Commercial Strategy**: By understanding the "Demand Variation" in our plots, we can minimize our exposure to aFRR activation costs. Accurate forecasting at the "Commercial Heart" ensures we stay within our Balancing Group equilibrium.  

### Virtual Power Plants (VPP) and Flexibility
**DGs (Distributed Generators)** connected to the distribution grid has enabled the deployment of Virtual Power Plants (VPP).  
VPP Operation: A VPP aggregates small assets (Solar, Wind, Battery) to act as a single large-scale flexible unit.  

**Technical Utility**: During the "Weekend Dip" identified in our findings, a VPP can provide negative control power (reducing generation or increasing consumption) to the distribution grid, monetizing the flexibility of assets when industrial demand is low.  

### Long-Term Hedging: PPAs and Solar Integration
To stabilize long-term procurement costs, we utilize PPAs (Power Purchase Agreements).  
**Solar PPA Alignment**: Solar generation naturally correlates with the mid-day portion of the Peak block. By signing a Solar PPA, we hedge the volume of the mid-day load spike seen in our daily profiles.  

**Volume Customization**: "How much you want to" procure depends on the Residual Load calculation. We use the 5-year average to determine the "P90" level of solar generation needed to cover the peak without over-procuring during the low-demand "Deep Dips" of the summer months.  

In [ ]:
df_recent = df[df['year'] >= 2021]

daily_profile = (
    df_recent.groupby(['hour', 'is_weekend'])['load_mwh']
    .mean()
    .reset_index()
)
daily_profile['day_type'] = daily_profile['is_weekend'].map({False: 'Weekday', True: 'Weekend'})

fig = px.line(
    daily_profile,
    x='hour',
    y='load_mwh',
    color='day_type',
    title="Average Daily Load Profile (2021-2026)",
    color_discrete_sequence=px.colors.sequential.Viridis
)
fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Load (MWh)",
    legend_title_text="Day Type"
)
fig.update_xaxes(tickmode='array', tickvals=list(range(24)))
fig.show()


In [ ]:
monthly = df[df['year'].between(2020, 2025)].groupby(['year', 'month'])['load_mwh'].mean().reset_index()

fig = px.line(
    monthly,
    x='month',
    y='load_mwh',
    color='year',
    markers=True,
    title="Yearly Consumption Comparison (Monthly Averages)",
    color_discrete_sequence=px.colors.qualitative.T10
)
fig.update_layout(yaxis_title="Average Load (MWh)", xaxis_title="Month")
fig.update_xaxes(
    tickmode='array',
    tickvals=list(range(1, 13)),
    ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
)
fig.show()

'''
Energy use is highest in winter.
It drops sharply in spring.
Summer has the lowest and most stable demand.
Consumption rises again in autumn.
Biggest jump = October and November.
'''


In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else: # 9 10 11
        return 'Autumn'

df['season'] = df['month'].apply(get_season)

season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
season_colors = {"Winter": "#1f77b4", "Spring": "#2ca02c", "Summer": "#ff7f0e", "Autumn": "#8c564b"}

season_profile = df.groupby(['season', 'hour'])['load_mwh'].mean().reset_index()

fig = px.line(
    season_profile,
    x='hour',
    y='load_mwh',
    color='season',
    category_orders={'season': season_order},
    color_discrete_map=season_colors,
    title="Daily Load Profile by Season (Seasonal Variations)"
)
fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Load (MWh)",
    legend_title_text="Season"
)
fig.update_xaxes(tickmode='array', tickvals=list(range(24)))
fig.show()

# average load per season for comparison
print("Average Load by Season:")
print(df.groupby('season')['load_mwh'].mean().reindex(season_order))


**COVID-19 Impact (2020–2021):** "Flattening of the Peak" = Lockdowns reduced the magnitude of the 8:00 AM industrial spike. Technically, this represents a shift from commercial-centric consumption to a more distributed, residential-weighted profile.

**Russia–Ukraine War & Energy Crisis (2022–Present):** Post-February 2022 data shows evidence of "Demand Destruction." High wholesale prices (influenced by the Gas Merit Order) triggered energy-saving measures across the German manufacturing sector.

In [ ]:
def categorize_era(year):
    if year in [2020, 2021]:
        return 'Covid (2020-2021)'
    elif year >= 2022:
        return 'Post-Covid (2022-2025)'
    else:
        return 'Pre-Covid' # not data for

df['era'] = df['year'].apply(categorize_era)

era_profile = df.groupby(['era', 'hour'])['load_mwh'].mean().reset_index()

fig = px.line(
    era_profile,
    x='hour',
    y='load_mwh',
    color='era',
    title="Covid vs. Post-Covid",
    color_discrete_sequence=['#e74c3c', '#27ae60']
)
fig.update_layout(xaxis_title="Hour of Day", yaxis_title="Load (MWh)")
fig.update_xaxes(tickmode='array', tickvals=list(range(24)))
fig.show()

# statistics
stats = df.groupby('era')['load_mwh'].mean()
print("Average Load Comparison:")
print(stats)


In [ ]:
war_start_date = pd.Timestamp('2022-02-24')

war_order = ['Pre-War (Stable Prices)', 'War Period (Energy Crisis)']

df['war_period'] = df.index.map(
    lambda x: war_order[0] if x < war_start_date else war_order[1]
)

war_profile = df.groupby(['war_period', 'hour'])['load_mwh'].mean().reset_index()

fig = px.line(
    war_profile,
    x='hour',
    y='load_mwh',
    color='war_period',
    category_orders={'war_period': war_order},
    color_discrete_sequence=['#3498db', '#e67e22'],
    title="Impact of Russia-Ukraine War on Energy Demand"
)
fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Load (MWh)",
    legend_title_text="Period"
)
fig.update_xaxes(tickmode='array', tickvals=list(range(24)))
fig.show()

stats = df.groupby('war_period')['load_mwh'].mean()
pre_war = stats[war_order[0]]
post_war = stats[war_order[1]]
drop_pct = ((pre_war - post_war) / pre_war) * 100

print(f"Average Load Pre-War:  {pre_war:.2f} MWh")
print(f"Average Load Post-War: {post_war:.2f} MWh")
print(f"Total Consumption Drop: {drop_pct:.2f}%")


In [ ]:
def fetch_and_plot_weather():
    openmeteo = openmeteo_requests.Client()

    # 2. Kassel (Center of Germany)
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 51.3127,
        "longitude": 9.4797,
        "start_date": "2020-01-01",
        "end_date": datetime.now().strftime('%Y-%m-%d'),
        "hourly": ["temperature_2m", "shortwave_radiation", "wind_speed_10m", "precipitation"],
        "timezone": "Europe/Berlin"
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]

    hourly = response.Hourly()
    
    temp = hourly.Variables(0).ValuesAsNumpy()
    solar = hourly.Variables(1).ValuesAsNumpy()
    wind = hourly.Variables(2).ValuesAsNumpy()
    precip = hourly.Variables(3).ValuesAsNumpy()

    start_dt = pd.to_datetime(hourly.Time(), unit="s", utc=True)
    time_index = pd.date_range(
        start=start_dt,
        periods=len(temp),
        freq=pd.Timedelta(seconds=hourly.Interval())
    )

    df_weather = pd.DataFrame({
        "Temperature (°C)": temp,
        "Solar Radiation (W/m²)": solar,
        "Wind Speed (km/h)": wind,
        "Precipitation (mm)": precip
    }, index=time_index)

    df_weather.index = df_weather.index.tz_convert("Europe/Berlin").tz_localize(None)

    df_daily = df_weather.resample('D').mean()

    colors = ['#ff7f0e', '#fdb813', '#1f77b4', '#2ca02c']
    columns = ["Temperature (°C)", "Solar Radiation (W/m²)", "Wind Speed (km/h)", "Precipitation (mm)"]

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=[f"Germany Average: {col} (2020 - Present)" for col in columns]
    )

    for i, col in enumerate(columns, start=1):
        fig.add_trace(
            go.Scatter(
                x=df_daily.index,
                y=df_daily[col],
                mode='lines',
                line=dict(color=colors[i - 1], width=1.5),
                name=col
            ),
            row=i,
            col=1
        )
        fig.update_yaxes(title_text=col, row=i, col=1)

    fig.update_layout(height=900, showlegend=False)
    fig.update_xaxes(title_text="Date", row=4, col=1)
    fig.show()

# Execute
fetch_and_plot_weather()
# adding monthly graphs = weather 
# merge = weather data and load data for a day


In [ ]:
CSV_FILE = "Actual_consumption_202001010000_202601010000_Quarterhour.csv"

def smard_csv(file_path):
    # Semicolon separator, comma = thousands, dot = decimal
    df = pd.read_csv(file_path, sep=';', thousands=',', decimal='.', index_col=False)
    return df
df_load = smard_csv(CSV_FILE)

df = df_load
df.head(100)
# generation and load profile


In [ ]:
CSV_FILE = "Gro_handelspreise_202001010000_202601010000_Viertelstunde.csv"

def smard_csv(file_path):
    # Semicolon separator, comma = thousands, dot = decimal
    df = pd.read_csv(file_path, sep=';', thousands=',', decimal='.', index_col=False)
    return df
df_load = smard_csv(CSV_FILE)

df = df_load
df.head(100)
# load profile and market price


In [ ]:
LOAD_CSV = "Actual_consumption_202001010000_202601010000_Quarterhour.csv"
PRICE_CSV = "Gro_handelspreise_202001010000_202601010000_Viertelstunde.csv"
DATE_TO_PLOT = "2023-05-21"  # <--- CHANGE THIS TO ANY DAY YOU WANT

# ==========================================
# 2. DATA PROCESSING FUNCTION
# ==========================================
def get_merged_data():
    # --- A. Process Load Data ---
    print("Reading Load Data...")
    df_l = pd.read_csv(LOAD_CSV, sep=';', thousands=',', decimal='.', index_col=False)
    df_l.columns = df_l.columns.str.strip() # Clean hidden spaces
    
    # Handle European Date Format
    df_l['datetime'] = pd.to_datetime(df_l['Start date'], dayfirst=True)
    df_l = df_l.set_index('datetime')
    
    load_col = 'grid load [MWh] Original resolutions'
    df_l = df_l[[load_col]].rename(columns={load_col: 'load_mwh'})
    df_l = df_l.resample('h').mean() # Downsample 15min -> 1hour

    # --- B. Process Price Data ---
    print("Reading Price Data...")
    df_p = pd.read_csv(PRICE_CSV, sep=';', thousands=',', decimal='.', index_col=False)
    df_p.columns = df_p.columns.str.strip()
    
    df_p['datetime'] = pd.to_datetime(df_p['Datum von'], dayfirst=True)
    df_p = df_p.set_index('datetime')
    
    price_col = 'Deutschland/Luxemburg [€/MWh] Originalauflösungen'
    # Convert to numeric and scale (4188 -> 41.88)
    df_p['price_eur'] = pd.to_numeric(df_p[price_col], errors='coerce') / 100.0
    df_p = df_p[['price_eur']].resample('h').mean()

    # --- C. Fetch Weather Data via API ---
    print("Fetching Weather Data...")
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # Use the date range from your files
    start_date = df_l.index.min().strftime('%Y-%m-%d')
    end_date = df_l.index.max().strftime('%Y-%m-%d')
    
    params = {
        "latitude": 51.3127, "longitude": 9.4797, # Kassel (Center of DE)
        "start_date": start_date, "end_date": end_date,
        "hourly": ["temperature_2m", "shortwave_radiation"],
        "timezone": "Europe/Berlin"
    }
    
    responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
    h_api = responses[0].Hourly()
    
    time_idx = pd.date_range(
        start=pd.to_datetime(h_api.Time(), unit="s", utc=True), 
        periods=len(h_api.Variables(0).ValuesAsNumpy()), 
        freq='h'
    )
    
    df_w = pd.DataFrame({
        "temp": h_api.Variables(0).ValuesAsNumpy(),
        "solar": h_api.Variables(1).ValuesAsNumpy()
    }, index=time_idx)
    
    # Align timezones
    df_w.index = df_w.index.tz_convert("Europe/Berlin").tz_localize(None)

    # --- D. FINAL MERGE ---
    print("Merging all datasets...")
    df_final = pd.merge(df_l, df_p, left_index=True, right_index=True, how='inner')
    df_final = pd.merge(df_final, df_w, left_index=True, right_index=True, how='inner')
    
    return df_final

# ==========================================
# 3. PLOTTING FUNCTION
# ==========================================
def plot_full_analysis(df, target_date):
    try:
        day_data = df.loc[target_date]
    except KeyError:
        print(f"Error: Date {target_date} not found in the data range.")
        return

    if day_data.empty:
        print("No data available for this date.")
        return

    hours = day_data.index.hour

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=[
            "Grid Load (MWh)",
            "Market Price (€/MWh)",
            "Temperature (°C)",
            "Solar Radiation (W/m²)"
        ]
    )

    # 1. Grid Load (MWh)
    fig.add_trace(
        go.Scatter(
            x=hours,
            y=day_data['load_mwh'],
            mode='lines+markers',
            line=dict(color='black', width=2),
            marker=dict(size=6)
        ),
        row=1,
        col=1
    )
    fig.update_yaxes(title_text="Load (MWh)", row=1, col=1)

    # 2. Market Price (€/MWh)
    fig.add_trace(
        go.Scatter(
            x=hours,
            y=day_data['price_eur'],
            mode='lines',
            line=dict(color='blue', width=2, shape='hv'),
            fill='tozeroy',
            fillcolor='rgba(0, 0, 255, 0.1)'
        ),
        row=2,
        col=1
    )
    fig.update_yaxes(title_text="Price (€/MWh)", row=2, col=1)

    # 3. Temperature (°C)
    fig.add_trace(
        go.Scatter(
            x=hours,
            y=day_data['temp'],
            mode='lines+markers',
            line=dict(color='red', width=2),
            marker=dict(symbol='square', size=6)
        ),
        row=3,
        col=1
    )
    fig.update_yaxes(title_text="Temp (°C)", row=3, col=1)

    # 4. Solar Radiation (W/m²)
    fig.add_trace(
        go.Scatter(
            x=hours,
            y=day_data['solar'],
            mode='lines',
            line=dict(color='orange', width=2),
            fill='tozeroy',
            fillcolor='rgba(255, 165, 0, 0.4)'
        ),
        row=4,
        col=1
    )
    fig.update_yaxes(title_text="Solar (W/m²)", row=4, col=1)

    fig.update_xaxes(title_text="Hour of Day", tickmode='array', tickvals=list(range(24)), row=4, col=1)
    fig.update_layout(title=f"Energy Market & Weather: {target_date}", height=900, showlegend=False)

    print(f"Plotting complete for {target_date}.")
    fig.show()

# ==========================================
# 4. RUN EVERYTHING
# ==========================================
df_all = get_merged_data()
plot_full_analysis(df_all, DATE_TO_PLOT)


In [ ]:
CSV_FILE = "EC_EV_dataset.xlsx - Load.csv"

def smard_csv(file_path):

    df = pd.read_csv(file_path, sep=',', decimal='.', index_col=0)
    return df

df_load = smard_csv(CSV_FILE)

print(df_load.head())

In [ ]:
CSV_FILE = "EC_EV_dataset.xlsx - PV.csv"

def smard_csv(file_path):

    df = pd.read_csv(file_path, sep=',', decimal='.', index_col=0)
    return df

df_load = smard_csv(CSV_FILE)

print(df_load.head())